# 02 - Data Cleaning & Missing Value Imputation

**Project:** CrediPredict – Loan Approval Prediction System  
**Author:** B.Tech Data Science Student  
**Objective:** Clean raw applicant records by imputing missing values, removing duplicates, inspecting data types, and handling extreme values appropriately.


### Step 1: Import Libraries and Load Raw Data

In [ ]:
import sys
import os
import pandas as pd
import numpy as np

# Add src folder to path
sys.path.append(os.path.abspath('..'))
from src.data_loader import load_raw_data

df = load_raw_data()
df.head()

### Step 2: Check for Duplicate Rows
Duplicates can lead to data leakage and biased model evaluation.

In [ ]:
duplicate_count = df.duplicated().sum()
print(f"Total Duplicate Rows found: {duplicate_count}")

### Step 3: Missing Values Imputation

#### Strategy & Interview Explanation:
1. **Categorical Columns** (`Gender`, `Married`, `Dependents`, `Self_Employed`, `Credit_History`):
   - We fill missing values with the **Mode** (most frequent category) because categorical labels cannot be averaged.
2. **Numerical Columns** (`LoanAmount`, `Loan_Amount_Term`):
   - We fill missing values using the **Median** rather than Mean because income and loan amounts exhibit right-skewed distributions, making the median robust against extreme outliers.

In [ ]:
df_cleaned = df.copy()

# 1. Categorical imputation (Mode)
cat_cols = ['Gender', 'Married', 'Dependents', 'Self_Employed', 'Credit_History']
for col in cat_cols:
    mode_val = df_cleaned[col].mode()[0]
    df_cleaned[col] = df_cleaned[col].fillna(mode_val)
    print(f"Filled missing values in '{col}' with Mode: {mode_val}")

# 2. Numerical imputation (Median)
num_cols = ['LoanAmount', 'Loan_Amount_Term']
for col in num_cols:
    median_val = df_cleaned[col].median()
    df_cleaned[col] = df_cleaned[col].fillna(median_val)
    print(f"Filled missing values in '{col}' with Median: {median_val}")

### Step 4: Verify Missing Value Removal

In [ ]:
remaining_nulls = df_cleaned.isnull().sum().sum()
print(f"Remaining Null Values across all columns: {remaining_nulls}")

### Step 5: Outlier Inspection

#### Interview Explanation:
In loan applications, high applicant incomes (e.g., $50,000+) are valid real-world occurrences (high earners applying for credit). Rather than blindly dropping these extreme data points, we keep them so the model learns realistic high-income applicant behavior.

In [ ]:
q1 = df_cleaned['ApplicantIncome'].quantile(0.25)
q3 = df_cleaned['ApplicantIncome'].quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr

high_earners = df_cleaned[df_cleaned['ApplicantIncome'] > upper_bound]
print(f"ApplicantIncome IQR Upper Bound: {upper_bound:.2f}")
print(f"High income applicant records count: {len(high_earners)} ({len(high_earners)/len(df_cleaned)*100:.2f}%)")

### Step 6: Save Cleaned Dataset

In [ ]:
processed_dir = os.path.join('..', 'data', 'processed')
os.makedirs(processed_dir, exist_ok=True)
cleaned_path = os.path.join(processed_dir, 'cleaned_loan_data.csv')

df_cleaned.to_csv(cleaned_path, index=False)
print(f"Cleaned dataset saved successfully to: {cleaned_path}")